# Tarea para el Hogar 02

Esta Tarea para el Hogar 02 se entrega el final de la segunda clase
<br> se espera de usted que intente avanzar con los desafios propuestos y que los traiga terminados para la Clase 03, ya que se analizarán los resultados

##  1. Ensembles de Modelos

Vea el siguiente video [BBC - The Code - The Wisdom of the Crowd](https://www.youtube.com/watch?v=iOucwX7Z1HU)    ( 5 min)


Lea los siguientes artículos


*   [The Wisdom of Crowds (Vox Populi) by Francis Galton](https://www.all-about-psychology.com/the-wisdom-of-crowds.html)  (10 min)
*   [A Gentle Introduction to Ensemble Learning](https://machinelearningmastery.com/what-is-ensemble-learning/)  (10 min)





---



##  2.  Zero2Hero   primera parte
Se han lanzado los primeros fascículos coleccionables llamados "from Zero to Hero" que muy detalladamente, paso a paso enseñan todo lo necesario de R para entender los scripts oficiales de la asignatura.
Están en el repositorio oficial de la asignatura, carpeta  **src/zero2hero**



---



## 3.  Grid Search

Busque en internet el precido significado de los hiperparámetros de la librería **rpart**  que está implementando el algoritmo **CART**  Classification and Regression Trees  propuesto en el año 1984 por Leo Brieman:

*   cp
*   maxdepth
*   minsplit
*   minbucket

Entienda que valores es razonable tome cada hiperparámetro,  en particular profundice en el hiperparámetro  **cp**  y la posibilidad que tome valores negativos.  Es válido consultar a su amigo de *capacidades especiales*  ChatGPT


En las siguientes celdas a un notebook incompleto, un esqueleto de codigo brindado a modo de facilitarle la tarea de codeo y permitir que su valiosa cognición se concentre temas conceptuales de Ciencia de Datos

Modifiquelo agregando loops para que recorra TODOS los hiperparámetros de rpart  < cp, maxdepth, minsplit, minbucket >, y luego póngalo a correr. Recuerde cambiar por SU semilla
Tenga muy presente la granularidad que eligirá para cada hiperparámetro.

### Seteo del ambiente local
Configuración de directorios de trabajo y descarga del dataset directamente al entorno local (sin requerir Google Drive ni credenciales de Kaggle).

In [1]:
# Determinar la raíz del proyecto de forma absoluta
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
setwd(dir_base)

# Crear carpetas locales para datasets y experimentos
dir.create(file.path(dir_base, "datasets"), showWarnings = FALSE, recursive = TRUE)
dir.create(file.path(dir_base, "exp"), showWarnings = FALSE, recursive = TRUE)

# Descargar el dataset si no existe localmente
url_dataset <- "https://storage.googleapis.com/open-courses/utn2026-b40a/dataset_pequeno.csv"
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

if (!file.exists(archivo_dataset)) {
  cat("Descargando dataset_pequeno.csv...\n")
  download.file(url_dataset, destfile = archivo_dataset, mode = "wb")
  cat("Descarga completada exitosamente.\n")
} else {
  cat("El dataset ya se encuentra disponible en:", archivo_dataset, "\n")
}

El dataset ya se encuentra disponible en: /workspace/datasets/dataset_pequeno.csv 


limpio el ambiente de R

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,658525,35.2,1454520,77.7,1420947,75.9
Vcells,1225758,9.4,8388608,64.0,1975054,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")
require("parallel")
if (!require("primes")) install.packages("primes")
require("primes")

Loading required package: data.table

Loading required package: rpart

Loading required package: parallel

Loading required package: primes



Aqui debe poner SU semiila primigenia

In [4]:
PARAM <- list()
# reemplazar por su primer semilla
PARAM$semilla_primigenia <- 115879
PARAM$qsemillas <- 1

PARAM$training_pct <- 70L  # entre  1L y 99L

# elegir SU dataset comentando/ descomentando
PARAM$dataset_nom <- "./datasets/dataset_pequeno.csv"

In [5]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
# particionar( data=dataset, division=c(70,30), agrupa=clase_ternaria, seed=semilla)
#   crea una particion 70, 30

particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}


In [6]:
ArbolEstimarGanancia <- function(semilla, training_pct, param_basicos) {
  # particiono estratificadamente el dataset
  particionar(dataset,
    division = c(training_pct, 100L -training_pct),
    agrupa = "clase_ternaria",
    seed = semilla # aqui se usa SU semilla
  )

  # genero el modelo
  # predecir clase_ternaria a partir del resto
  modelo <- rpart("clase_ternaria ~ .",
    data = dataset[fold == 1], # fold==1  es training,  el 70% de los datos
    xval = 0,
    control = param_basicos
  ) # aqui van los parametros del arbol

  # aplico el modelo a los datos de testing
  prediccion <- predict(modelo, # el modelo que genere recien
    dataset[fold == 2], # fold==2  es testing, el 30% de los datos
    type = "prob"
  ) # type= "prob"  es que devuelva la probabilidad

  # prediccion es una matriz con TRES columnas,
  #  llamadas "BAJA+1", "BAJA+2"  y "CONTINUA"
  # cada columna es el vector de probabilidades


  # calculo la ganancia en testing  qu es fold==2
  ganancia_test <- dataset[
    fold == 2,
    sum(ifelse(prediccion[, "BAJA+2"] > 0.025,
      ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
      0
    ))
  ]

  # escalo la ganancia como si fuera todo el dataset
  ganancia_test_normalizada <- ganancia_test / (( 100 - PARAM$training_pct ) / 100 )

  return(
    c( list("semilla" = semilla),
      param_basicos,
      list( "ganancia_test" = ganancia_test_normalizada )
     )
  )
}


In [7]:
ArbolesMontecarlo <- function(semillas, param_basicos) {

  # la funcion mcmapply  llama a la funcion ArbolEstimarGanancia
  #  tantas veces como valores tenga el vector  PARAM$semillas
  salida <- mcmapply(ArbolEstimarGanancia,
    semillas, # paso el vector de semillas
    MoreArgs = list(PARAM$training_pct, param_basicos), # aqui paso el segundo parametro
    SIMPLIFY = FALSE,
    mc.cores = detectCores()
  )

  return(salida)
}


In [8]:
# carpeta de trabajo local
# por favor cambiar numero de experimento si se cambia el loop principal
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
experimento <- "HT2902"
dir_experimento <- file.path(dir_base, "exp", experimento)
dir.create(dir_experimento, showWarnings = FALSE, recursive = TRUE)
setwd(dir_experimento)

In [9]:
# lectura del dataset desde la carpeta local
find_project_root <- function() {
  curr <- normalizePath(getwd(), winslash = "/")
  while (curr != "/" && curr != dirname(curr)) {
    if (file.exists(file.path(curr, "Dockerfile")) || dir.exists(file.path(curr, "src"))) {
      return(curr)
    }
    curr <- dirname(curr)
  }
  return(normalizePath(getwd(), winslash = "/"))
}

dir_base <- if (dir.exists("/workspace")) "/workspace" else find_project_root()
archivo_dataset <- file.path(dir_base, "datasets", "dataset_pequeno.csv")

dataset <- fread(archivo_dataset)

# trabajo solo con los datos con clase, es decir 202107
dataset <- dataset[clase_ternaria != ""]

In [10]:

# genero numeros primos
primos <- generate_primes(min = 100000, max = 1000000)
set.seed(PARAM$semilla_primigenia) # inicializo
# me quedo con PARAM$qsemillas   semillas
PARAM$semillas <- sample(primos, PARAM$qsemillas )


In [11]:
# genero la data.table donde van los resultados detallados del Grid Search
# un registro para cada combinacion de < semilla, parametros >

if(file.exists("gridsearch_detalle.txt")){
  tb_grid_search_detalle <- fread("gridsearch_detalle.txt")
}else{
  tb_grid_search_detalle <- data.table(
    semilla = integer(),
    cp = numeric(),
    maxdepth = integer(),
    minsplit = integer(),
    minbucket = integer(),
    ganancia_test = numeric()
  )
}

nrow( tb_grid_search_detalle )

[1] 0

Esta es la parte del código que usted debe expandir a TODOS los hiperparámetros de rpart,
<br>ya que actualmente apenas recorre  maxdepth y  minsplit  dejando fijos  cp=-0.5  y minbucket=5

In [12]:

# itero por los loops anidados para cada hiperparametro
iter <- 0

for (vmax_depth in c(9)) {
  for (vmin_split in c(900)) {
    # notar como se agrega

    iter <- iter + 1
    cat( iter, " " )
    flush.console()
    if( iter*PARAM$qsemillas < nrow(tb_grid_search_detalle)+1 ) next

    # vminsplit  minima cantidad de registros en un nodo para hacer el split
    param_basicos <- list(
      "cp" = -1, # complejidad minima
      "maxdepth" = vmax_depth, # profundidad máxima del arbol
      "minsplit" = vmin_split, # tamaño minimo de nodo para hacer split
      "minbucket" = 225 # minima cantidad de registros en una hoja
    )

    # Un solo llamado, con la semilla 17
    ganancias <- ArbolesMontecarlo(PARAM$semillas, param_basicos)

    # agrego a la tabla
    tb_grid_search_detalle <- rbindlist(
      list( tb_grid_search_detalle,
            rbindlist(ganancias) )
    )

  }

  # grabo cada vez TODA la tabla en el loop mas externo
  fwrite( tb_grid_search_detalle,
          file = "gridsearch_detalle.txt",
          sep = "\t" )
}


1  

In [13]:
fwrite( tb_grid_search_detalle,
   file = "gridsearch_detalle.txt",
   sep = "\t"
)

In [14]:
# cantidad de registros de la tabla
nrow(tb_grid_search_detalle)

[1] 1

In [15]:
# muestro la tabla
tb_grid_search_detalle

semilla,cp,maxdepth,minsplit,minbucket,ganancia_test
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
409483,-1,9,900,225,511833333


In [16]:
# genero y grabo el resumen
tb_grid_search <- tb_grid_search_detalle[,
  list( "ganancia_mean" = mean(ganancia_test),
    "qty" = .N ),
  list( cp, maxdepth, minsplit, minbucket )
]


In [17]:
# ordeno descendente por ganancia
setorder( tb_grid_search, -ganancia_mean )


In [18]:
# veo los 10 mejores hiperparámetros
tb_grid_search[1:10]

cp,maxdepth,minsplit,minbucket,ganancia_mean,qty
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
-1,9,900,225,511833333,1
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA
NA,NA,NA,NA,NA,NA


In [19]:
# genero un id a la tabla
tb_grid_search[, id := .I ]

fwrite( tb_grid_search,
  file = "gridsearch.txt",
  sep = "\t"
)


# 4.  Análisis de resultados de Grid Search

La salida de la corrida anterior queda en `./exp/HT2900`
<br>HT significa Hyperparameter Tuning
<br>El Grid Search es un método de fuerza bruta de un altísimo costo computacional.
<br>Queremos ver si es posible crear un algoritmo de optimización de hiperparámetros que se ahorre recorrer ciertas porciones muy malas del espacio de búsqueda. Algo del estilo “cada vez que pruebo una combinación de hiperparámetros donde  cp > 1 , la ganancia es muy mala, con lo cual ni vale la pena perder el tiempo explorando en esa region”

<br>Levante el archivo de salida gridsearch.txt  a una planilla tipo Excel y analícelo detenidamente
<br>Ordene por ganancia_mean descendente
<br>
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch  cargue el mejor del ranking en la posición 1, el segundo en la 2, y el 5, 10, 50 y 100. Es decir debe cargr SEIS lineas en las celdas correspondientes a su nombre.
<br>
<br>Verifique que efectivamente está dado de alta en la competencia Kaggle  "Data Mining, Inicial 2026 B"  lo que debio haber hecho siguiendo el capítulo Arranque en Frio de El Libro de la Asignatura
<br>En la Planilla Colaborativa, Hoja  C3-GridSearch, utilizando el notebook  **src/arboles/z102_FinalTrain.ipynb**  haga el submit a Kaggle de cada una de las SEIS combinaciones de hiperparámetros y completa la columna Public Leaderboard

<br>
<br>El de mayor ganancia_mean  decimos que es el primero del ranking
En Zulip, correspondiente channel  #Tarea Hogar 02 , topic Analisis Grid Search   intente contestar estas preguntas:

* ¿Qué combinaciones de hiperparámetros poseen una ganancia muy buena?
* ¿Hay algun hiperparámetro que para cierto valor siempre genera una ganancia muy mala, a independientemente de lo que valgan los otros hiperparámetros ?
* ¿Que combinaciones de hiperparámetros es pésima y hubiera sido bueno ahorrarse esas corridas ?

( tiempo estimado 40 minutos, dificultad media )